In [1]:
import os, sys

REPO = "BondingIssues"
URL = f"https://github.com/jomana2006/BondingIssues.git"

if not os.path.exists(f"/content/BondingIssues/O.G-GeomSAC"):
    !git clone {URL}
else:
    !git -C /content/BondingIssues/O.G-GeomSAC pull

sys.path.append(f"/content/BondingIssues/O.G-GeomSAC")

Cloning into 'BondingIssues'...
remote: Enumerating objects: 133, done.
remote: Counting objects: 100% (133/133), done.
remote: Compressing objects: 100% (131/131), done.
remote: Total 133 (delta 92), reused 3 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (133/133), 36.84 KiB | 2.83 MiB/s, done.
Resolving deltas: 100% (92/92), done.


In [2]:
%run "/content/BondingIssues/O.G-GeomSAC/Utils.ipynb"
%run "/content/BondingIssues/O.G-GeomSAC/NeuralNetwork.ipynb"
%run "/content/BondingIssues/O.G-GeomSAC/buffer.ipynb"
%run "/content/BondingIssues/O.G-GeomSAC/MolGraphEnv.ipynb"

Already up to date.
Already up to date.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 8.1 MB/s  0:00:00


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


fatal: destination path 'BondingIssues' already exists and is not an empty directory.


/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Already up to date.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.8/31.8 MB 31.0 MB/s  0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of dive-into-graphs to determine which version is compatible with other requirements. This could take a while.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 62.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 73.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 152.2 MB/s  0:00:00
  Created wheel for typed-argument-parser: filename=typed_argument_parser-1.5.4-py3-none-any.whl size=18340 sha256=6113e61a1b3d9d5d973ff49b0b68ca115fb4891f64dc52a2f666a5666259e95a
  Stored in directory: /root/.cache/pip/wheels/f0/e4/2f/8bbc1369f449f3f9fa2f0d

In [3]:
import torch.optim as optim

In [4]:
class SoftActorCriticAgent:
    def __init__(self, env, state):
        torch.autograd.set_detect_anomaly(True)
        self.env = env
        self.n_actions = sum(env.action_space.nvec)
        self.action_space = env.action_space.nvec.tolist()
        self.ac_loss, self.q_loss = [], []

        self.critic_v = StateValueNetwork(state)
        self.critic_v_target = StateValueNetwork(state)
        self.critic_q_1 = ActionValueNetwork(state, self.n_actions)
        self.critic_q_2 = ActionValueNetwork(state, self.n_actions)
        self.actor = PolicyNetwork(env, state, self.n_actions)

        self.actor_optim = optim.Adam(self.actor.parameters(), lr=3 * 10e-4)  # 0.003
        self.v_optim = optim.Adam(self.critic_v.parameters(), lr=0.003)
        self.q1_optim = optim.Adam(self.critic_q_1.parameters(), lr=0.003)
        self.q2_optim = optim.Adam(self.critic_q_2.parameters(), lr=0.003)

        self.gamma = 0.99
        self.tau = 0.005
        self.batch_size = 32
        self.reward_scale = 10
        self.replay_buffer = ReplayBuffer(self.batch_size)
        self.update_target(1)

    def select_actions(self, state, reparam=True):
        self.actor.train()
        with torch.no_grad():
            probabilities, actions, log_p = self.actor.sample(state, self.action_space)
        return probabilities, actions, log_p

    def train(self):
        if len(self.replay_buffer.replay_buffer) < self.batch_size:
            return

        samples = self.replay_buffer.sample()
        states = samples[0]
        probabilities = samples[1]
        rewards = samples[2]
        next_states = samples[3]
        dones = samples[4]

        current_q_1 = self.critic_q_1(states, probabilities)
        current_q_2 = self.critic_q_2(states, probabilities)
        current_critic_v = self.critic_v(states)

        _, _, log_p = self.actor.sample(states, self.env.action_space.nvec.tolist())

        target_q = rewards * self.reward_scale + (self.gamma * self.critic_v_target(next_states) * (1 - dones))

        q1_loss = F.mse_loss(current_q_1, target_q.detach())
        q2_loss = F.mse_loss(current_q_2, target_q.detach())
        self.q_loss.append((q1_loss, q2_loss))

        self.q1_optim.zero_grad()
        q1_loss.backward()
        self.q1_optim.step()
        self.q2_optim.zero_grad()
        q2_loss.backward()
        self.q2_optim.step()

        q1 = self.critic_q_1(states, probabilities)
        q2 = self.critic_q_2(states, probabilities)

        predicted_new_q = torch.min(q1, q2)
        target_critic_v = predicted_new_q - log_p.unsqueeze(1)
        critic_loss = F.mse_loss(current_critic_v, target_critic_v.detach())
        self.v_optim.zero_grad()
        critic_loss.backward()
        self.v_optim.step()

        actor_loss = (log_p * (log_p - predicted_new_q)).mean()
        self.ac_loss.append(actor_loss)

        self.actor_optim.zero_grad()
        actor_loss.backward()
        self.actor_optim.step()
        self.update_target(self.tau)

    def update_target(self, tau):
        for target_param, param in zip(self.critic_v_target.parameters(), self.critic_v.parameters()):
            target_param.data.copy_(tau * param.data + (1 - tau) * target_param.data)
